# 01 — Landmark-safe data cleaning

This notebook creates the cleaned visit-level inputs for the final Parkinson's disease fall-risk workflow. It follows the revision extraction structure wherever that logic remains defensible and applies the approved corrections from the updated-data audit.

The notebook stops before patient-level aggregation, imputation, feature engineering, feature selection, and modeling. It never exports the fall outcome with predictor tables.

## How to use this notebook

Run every cell from top to bottom in a fresh kernel. The displayed outputs are aggregate checks only; no patient records are printed. The final comparison must show exact agreement with the verified internal cleaning outputs before Notebook 02 begins aggregation.

## 1. Set up paths and libraries

Load the small set of required libraries and locate the project root from any folder inside the repository.

In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

working_directory = Path.cwd().resolve()
project_root = next(
    (
        folder
        for folder in [working_directory, *working_directory.parents]
        if (folder / "AGENTS.md").is_file()
        and (folder / "docs/research_protocol.md").is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError("Launch this notebook from within the project directory.")

print("Project root:", project_root)

Project root: /Users/rafsan_temp/Library/CloudStorage/OneDrive-SeattleUniversity/SU Projects/pd-fall-risk


Create a new output location for the publication workflow. The verified Notebook 22 outputs remain unchanged and serve as the reproducibility reference.

In [2]:
data_directory = project_root / "data"
reference_directory = project_root / "results/final_pipeline/02_cleaning/cleaned_visits"
output_directory = (
    project_root
    / "results/final_pipeline/06_final_fit_and_performance_summary/01_data_cleaning"
)
cleaned_directory = output_directory / "cleaned_visits"
cleaned_directory.mkdir(parents=True, exist_ok=True)

print("Raw data:", data_directory.relative_to(project_root))
print("New outputs:", cleaned_directory.relative_to(project_root))
print("Verified reference:", reference_directory.relative_to(project_root))

Raw data: data
New outputs: results/final_pipeline/06_final_fit_and_performance_summary/01_data_cleaning/cleaned_visits
Verified reference: results/final_pipeline/02_cleaning/cleaned_visits


## 2. Reconstruct patient-specific landmarks

Retain the revision cohort definition: Parkinson's disease participants who are enrolled, complete, or withdrawn because of death. The latest dated, observed `FLNFR12M` record establishes the outcome month; the index month is exactly 12 calendar months earlier.

In [3]:
eligible_statuses = ["Enrolled", "Complete", "Withdraw Deceased"]

status = pd.read_csv(
    data_directory / "Participant_Status_24Aug2026.csv",
    dtype={"PATNO": "string"},
)
falls = pd.read_csv(
    data_directory / "Determination_of_Freezing_and_Falls_24Aug2026.csv",
    dtype={"PATNO": "string"},
)

cohort_ids = status.loc[
    status["COHORT"].eq(1)
    & status["ENROLL_STATUS"].isin(eligible_statuses),
    "PATNO",
]
cohort_falls = falls.loc[falls["PATNO"].isin(cohort_ids)].copy()

print(f"Eligible PD participants: {cohort_ids.nunique():,}")
print(f"Their fall-assessment records: {len(cohort_falls):,}")

Eligible PD participants: 1,362
Their fall-assessment records: 4,545


Parse the outcome month, select the latest observed outcome for each patient, and stop if the latest month is tied. The outcome value is used only to establish eligibility and timing in this notebook.

In [4]:
cohort_falls["outcome_date"] = pd.to_datetime(
    cohort_falls["INFODT"],
    format="%m/%Y",
    errors="coerce",
)

observed_outcomes = cohort_falls.dropna(
    subset=["FLNFR12M", "outcome_date"]
).copy()
latest_month = observed_outcomes.groupby("PATNO")["outcome_date"].transform("max")
latest_outcomes = observed_outcomes.loc[
    observed_outcomes["outcome_date"].eq(latest_month)
].copy()

assert latest_outcomes["PATNO"].is_unique, (
    "A patient has multiple observed outcomes in the latest month. "
    "Review the tied records before continuing."
)

landmarks = latest_outcomes[["PATNO", "outcome_date"]].copy()
landmarks["index_date"] = landmarks["outcome_date"] - pd.DateOffset(months=12)

assert len(landmarks) == 1_280
print(f"Patients with an observed outcome and landmark: {len(landmarks):,}")

Patients with an observed outcome and landmark: 1,280


## 3. Declare the predictor sources

The source contract lists only the fields required for the final 26-feature universe, its component variables (including the eight Neuro-QoL mobility items used solely to build the Gaussian score), and the prespecified sensitivity representations. Splitting the declaration into short cells keeps it readable without changing the source scope.

In [5]:
source_specs = {
    "demographics": {
        "file": "Demographics_24Aug2026.csv",
        "fields": [
            "BIRTHDT", "SEX", "RAASIAN", "RABLACK", "RAHAWOPI",
            "RAINDALS", "RANOS", "RAWHITE", "RAUNKNOWN",
        ],
    },
    "family_history": {
        "file": "Family_History_24Aug2026.csv",
        "fields": ["ANYFAMPD"],
    },
    "diagnosis_history": {
        "file": "PD_Diagnosis_History_24Aug2026.csv",
        "fields": [
            "PDDXDT", "DXTREMOR", "DXRIGID", "DXBRADY", "DXPOSINS", "DOMSIDE",
        ],
    },
    "dopamine_therapy": {
        "file": "Initiation_of_Dopaminergic_Therapy_24Aug2026.csv",
        "fields": ["DOPTHERST"],
    },
}

Add the cognitive, mood, autonomic, freezing, and mobility instruments used by the final candidates or their documented component representations.

In [6]:
source_specs.update({
    "moca": {
        "file": "Montreal_Cognitive_Assessment__MoCA__24Aug2026.csv",
        "fields": ["MCATOT"],
    },
    "freezing": {
        "file": "Determination_of_Freezing_and_Falls_24Aug2026.csv",
        "fields": ["FRZGT12M"],
    },
    "scopa_aut": {
        "file": "SCOPA-AUT_24Aug2026.csv",
        "fields": ["SCAU14", "SCAU16"],
    },
    "neuroqol_mobility": {
        "file": "Neuro_QoL__Lower_Extremity_Function__Mobility__-_Short_Form_24Aug2026.csv",
        "fields": [
            "NQMOB37", "NQMOB30", "NQMOB26", "NQMOB32",
            "NQMOB25", "NQMOB33", "NQMOB31", "NQMOB28",
        ],
    },
    "gds15": {
        "file": "Geriatric_Depression_Scale__Short_Version__24Aug2026.csv",
        "fields": [
            "GDSSATIS", "GDSDROPD", "GDSEMPTY", "GDSBORED", "GDSGSPIR",
            "GDSAFRAD", "GDSHAPPY", "GDSHLPLS", "GDSHOME", "GDSMEMRY",
            "GDSALIVE", "GDSWRTLS", "GDSENRGY", "GDSHOPLS", "GDSBETER",
        ],
    },
})

Add the MDS-UPDRS, other clinical-history, and vital-sign sources. Part III examination-state metadata is retained for the later ON/OFF sensitivity analysis.

In [7]:
source_specs.update({
    "updrs_part_i": {
        "file": "MDS-UPDRS_Part_I_24Aug2026.csv",
        "fields": [
            "NP1COG", "NP1HALL", "NP1DPRS", "NP1ANXS",
            "NP1APAT", "NP1DDS", "NP1RTOT",
        ],
    },
    "updrs_part_i_patient": {
        "file": "MDS-UPDRS_Part_I_Patient_Questionnaire_24Aug2026.csv",
        "fields": [
            "NP1SLPN", "NP1SLPD", "NP1PAIN", "NP1URIN",
            "NP1CNST", "NP1LTHD", "NP1FATG", "NP1PTOT",
        ],
    },
    "updrs_part_iii": {
        "file": "MDS-UPDRS_Part_III_24Aug2026.csv",
        "fields": ["PDSTATE", "NP3GAIT", "NP3PSTBL", "NHY", "NP3TOT"],
        "extra_metadata": [
            "EXAMDT", "EXAMTM", "PDTRTMNT", "PDMEDYN", "ONOFFORDER",
        ],
    },
    "updrs_part_iv": {
        "file": "MDS-UPDRS_Part_IV__Motor_Complications_24Aug2026.csv",
        "fields": [
            "NP4WDYSK", "NP4DYSKI", "NP4OFF", "NP4FLCTI",
            "NP4FLCTX", "NP4DYSTN", "NP4TOT",
        ],
    },
    "other_clinical": {
        "file": "Other_Clinical_Features_24Aug2026.csv",
        "fields": ["FEATPOSHYP", "FEATSUGRBD"],
    },
    "vital_signs": {
        "file": "Vital_Signs_24Aug2026.csv",
        "fields": [
            "HTCM", "WGTKG", "SYSSUP", "DIASUP", "HRSUP",
            "SYSSTND", "DIASTND", "HRSTND",
        ],
    },
})

Verify the source contract before reading complete tables. This catches a missing file or renamed field with a direct message.

In [8]:
contract_rows = []
for source_name, spec in source_specs.items():
    source_path = data_directory / spec["file"]
    if not source_path.is_file():
        raise FileNotFoundError(source_path)

    columns = pd.read_csv(source_path, nrows=0, encoding="utf-8-sig").columns
    required = {"REC_ID", "PATNO", "INFODT", *spec["fields"]}
    required.update(spec.get("extra_metadata", []))
    missing_fields = sorted(required.difference(columns))

    contract_rows.append({
        "source": source_name,
        "file": spec["file"],
        "required_fields": len(required),
        "missing_fields": ", ".join(missing_fields),
        "contract_passed": not missing_fields,
    })

source_contract = pd.DataFrame(contract_rows)
assert source_contract["contract_passed"].all(), source_contract.loc[
    ~source_contract["contract_passed"]
]

display(source_contract[["source", "required_fields", "contract_passed"]])

,source,required_fields,contract_passed
0,demographics,12,True
1,family_history,4,True
2,diagnosis_history,9,True
3,dopamine_therapy,4,True
4,moca,4,True
5,freezing,4,True
6,scopa_aut,5,True
7,neuroqol_mobility,11,True
8,gds15,18,True
9,updrs_part_i,10,True


## 4. Apply the landmark before cleaning

The temporal filter is applied before any range check, consistency correction, or history-based rule. Later records therefore cannot influence an earlier prediction profile.

In [9]:
def load_landmark_eligible_source(spec):
    """Load one source and retain dated records at or before each landmark."""
    source = pd.read_csv(
        data_directory / spec["file"],
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
    )
    source["PATNO"] = source["PATNO"].astype("string")
    source["DATE"] = pd.to_datetime(
        source["INFODT"],
        format="%m/%Y",
        errors="coerce",
    )

    nonblank_dates = source["INFODT"].str.strip().ne("")
    assert not (nonblank_dates & source["DATE"].isna()).any(), spec["file"]

    linked = source.merge(
        landmarks[["PATNO", "index_date"]],
        on="PATNO",
        how="inner",
    )
    eligible = linked["DATE"].notna() & linked["DATE"].le(linked["index_date"])

    flow = {
        "source": spec["file"],
        "raw_records": len(source),
        "linked_cohort_records": len(linked),
        "eligible_records": int(eligible.sum()),
        "later_records_excluded": int(linked["DATE"].gt(linked["index_date"]).sum()),
        "undated_records_excluded": int(linked["DATE"].isna().sum()),
        "eligible_patients": linked.loc[eligible, "PATNO"].nunique(),
    }
    return linked.loc[eligible].copy(), flow

Apply the same temporal helper to every declared source and retain a compact audit of included and excluded records.

In [10]:
eligible_sources = {}
temporal_flow_rows = []

for source_name, spec in source_specs.items():
    eligible_sources[source_name], flow = load_landmark_eligible_source(spec)
    temporal_flow_rows.append(flow)

temporal_flow = pd.DataFrame(temporal_flow_rows)
display(
    temporal_flow[[
        "source", "eligible_records", "later_records_excluded",
        "undated_records_excluded", "eligible_patients",
    ]]
)

,source,eligible_records,later_records_excluded,undated_records_excluded,eligible_patients
0,Demographics_24Aug2026.csv,1043,237,0,1043
1,Family_History_24Aug2026.csv,1193,267,164,1021
2,PD_Diagnosis_History_24Aug2026.csv,1043,233,0,1043
3,Initiation_of_Dopaminergic_Therapy_24Aug2026.csv,1245,565,0,838
4,Montreal_Cognitive_Assessment__MoCA__24Aug2026...,4984,1535,0,1040
5,Determination_of_Freezing_and_Falls_24Aug2026.csv,2944,1601,0,977
6,SCOPA-AUT_24Aug2026.csv,5421,1585,0,1040
7,Neuro_QoL__Lower_Extremity_Function__Mobility_...,2944,1572,0,976
8,Geriatric_Depression_Scale__Short_Version__24A...,5421,1579,0,1040
9,MDS-UPDRS_Part_I_24Aug2026.csv,9320,2229,0,1040


## 5. Define transparent conversion helpers

Load the annotated project code list so categorical codes are translated to their recorded meanings rather than treated as ordered numbers.

In [11]:
code_list_path = (
    project_root
    / "archive/legacy_project/CodeBook/Code_List_-__Annotated__23May2025.csv"
)
code_list = pd.read_csv(
    code_list_path,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig",
)

print(f"Annotated code-list rows: {len(code_list):,}")

Annotated code-list rows: 4,245


Define categorical conversion helpers that reject unexpected nonblank codes and preserve valid labels such as `Unknown` and `Uncertain`.

In [12]:
def code_labels(field):
    rows = code_list.loc[
        code_list["ITM_NAME"].eq(field),
        ["CODE", "DECODE"],
    ].drop_duplicates()
    return dict(zip(rows["CODE"].str.strip(), rows["DECODE"].str.strip()))


def label_category(series, field):
    raw = series.str.strip().replace("", pd.NA)
    mapping = code_labels(field)
    unexpected = raw.notna() & ~raw.isin(mapping)
    assert not unexpected.any(), f"Unexpected {field} code"
    return raw.map(mapping).astype("string")

Define a numeric conversion helper that leaves blanks missing and fails on any nonblank value that cannot be parsed.

In [13]:
def numeric_fields(frame, fields):
    converted = frame.copy()
    for field in fields:
        raw = converted[field].str.strip().replace("", pd.NA)
        numeric = pd.to_numeric(raw, errors="coerce")
        assert not (raw.notna() & numeric.isna()).any(), field
        converted[field] = numeric
    return converted

print("Conversion helpers are ready; no source values changed in this section.")

Conversion helpers are ready; no source values changed in this section.


## 6. Clean categorical and ordinary numeric fields

Clean demographics first. Birth month remains a date, sex remains nominal, and race indicators remain numeric source fields.

In [14]:
cleaned = {}

demographics = eligible_sources["demographics"].copy()
demographics["BIRTHDT"] = pd.to_datetime(
    demographics["BIRTHDT"].replace("", pd.NA),
    format="%m/%Y",
    errors="coerce",
)
demographics["SEX"] = label_category(demographics["SEX"], "SEX")

race_fields = [
    "RAASIAN", "RABLACK", "RAHAWOPI", "RAINDALS",
    "RANOS", "RAWHITE", "RAUNKNOWN",
]
cleaned["demographics"] = numeric_fields(demographics, race_fields)

Clean family and diagnosis history. The corrected diagnosis field is `DXPOSINS`, which actually represents postural instability at diagnosis.

In [15]:
family_history = eligible_sources["family_history"].copy()
family_history["ANYFAMPD"] = label_category(
    family_history["ANYFAMPD"],
    "ANYFAMPD",
)
cleaned["family_history"] = family_history

diagnosis_history = eligible_sources["diagnosis_history"].copy()
diagnosis_history["PDDXDT"] = pd.to_datetime(
    diagnosis_history["PDDXDT"].replace("", pd.NA),
    format="%m/%Y",
    errors="coerce",
)
for field in ["DXTREMOR", "DXRIGID", "DXBRADY", "DXPOSINS", "DOMSIDE"]:
    diagnosis_history[field] = label_category(diagnosis_history[field], field)
cleaned["diagnosis_history"] = diagnosis_history

Clean the remaining categorical clinical-history fields. Missing stays different from a recorded `No`, and `Uncertain` is retained as a real category.

In [16]:
dopamine_therapy = eligible_sources["dopamine_therapy"].copy()
dopamine_therapy["DOPTHERST"] = label_category(
    dopamine_therapy["DOPTHERST"],
    "DOPTHERST",
)
cleaned["dopamine_therapy"] = dopamine_therapy

other_clinical = eligible_sources["other_clinical"].copy()
for field in ["FEATPOSHYP", "FEATSUGRBD"]:
    other_clinical[field] = label_category(other_clinical[field], field)
cleaned["other_clinical"] = other_clinical

print(
    "FEATPOSHYP Uncertain records preserved:",
    int(other_clinical["FEATPOSHYP"].eq("Uncertain").sum()),
)
print(
    "FEATSUGRBD Uncertain records preserved:",
    int(other_clinical["FEATSUGRBD"].eq("Uncertain").sum()),
)

FEATPOSHYP Uncertain records preserved: 232
FEATSUGRBD Uncertain records preserved: 349


Convert the ordinary numeric instruments without filling missing values. Instrument-specific corrections are handled in the next sections.

In [17]:
ordinary_numeric_sources = [
    "moca",
    "freezing",
    "scopa_aut",
    "neuroqol_mobility",
    "updrs_part_i_patient",
    "updrs_part_iv",
]

for source_name in ordinary_numeric_sources:
    cleaned[source_name] = numeric_fields(
        eligible_sources[source_name],
        source_specs[source_name]["fields"],
    )

print(f"Ordinary numeric sources converted: {len(ordinary_numeric_sources)}")

Ordinary numeric sources converted: 6


## 7. Handle codebook-defined `101 = unable to rate`

For the audited rater-completed Part I fields, convert `101` to missing. This corrects the clinical meaning; it is not statistical imputation.

In [18]:
part_i = numeric_fields(
    eligible_sources["updrs_part_i"],
    source_specs["updrs_part_i"]["fields"],
)
part_i_items = [
    "NP1COG", "NP1HALL", "NP1DPRS", "NP1ANXS", "NP1APAT", "NP1DDS",
]
part_i_101_counts = {
    field: int(part_i[field].eq(101).sum())
    for field in part_i_items
}

for field in part_i_items:
    part_i[field] = part_i[field].replace(101, np.nan)

cleaned["updrs_part_i"] = part_i
print("Part I 101 values removed:", sum(part_i_101_counts.values()))

Part I 101 values removed: 32


Apply the same codebook rule to the six audited Part IV items. The Part IV total is retained as recorded and later aggregation handles form-level meaning.

In [19]:
part_iv = cleaned["updrs_part_iv"].copy()
part_iv_items = [
    "NP4WDYSK", "NP4DYSKI", "NP4OFF",
    "NP4FLCTI", "NP4FLCTX", "NP4DYSTN",
]
part_iv_101_counts = {
    field: int(part_iv[field].eq(101).sum())
    for field in part_iv_items
}

for field in part_iv_items:
    part_iv[field] = part_iv[field].replace(101, np.nan)

cleaned["updrs_part_iv"] = part_iv
print("Part IV 101 values removed:", sum(part_iv_101_counts.values()))

Part IV 101 values removed: 65


For Part III gait, postural stability, and Hoehn–Yahr, preserve a record-level assessment-status flag before clearing `101`. The flags are available only for the later sensitivity analysis.

In [20]:
part_iii = eligible_sources["updrs_part_iii"].copy()
part_iii["PDSTATE"] = label_category(part_iii["PDSTATE"], "PDSTATE")
part_iii = numeric_fields(
    part_iii,
    ["NP3GAIT", "NP3PSTBL", "NHY", "NP3TOT"],
)

part_iii_101_counts = {}
for field in ["NP3GAIT", "NP3PSTBL", "NHY"]:
    flag = f"{field}_WAS_101"
    part_iii_101_counts[field] = int(part_iii[field].eq(101).sum())
    part_iii[flag] = np.where(
        part_iii[field].isna(),
        np.nan,
        part_iii[field].eq(101).astype(float),
    )
    part_iii[field] = part_iii[field].replace(101, np.nan)

cleaned["updrs_part_iii"] = part_iii

Summarize Part III assessment-status handling without displaying any individual record.

In [21]:
part_iii_flags = [
    "NP3GAIT_WAS_101",
    "NP3PSTBL_WAS_101",
    "NHY_WAS_101",
]
affected_part_iii_patients = part_iii.loc[
    part_iii[part_iii_flags].eq(1).any(axis=1),
    "PATNO",
].nunique()

print("Part III 101 values removed:", sum(part_iii_101_counts.values()))
print("Patients with any eligible Part III 101:", affected_part_iii_patients)

Part III 101 values removed: 1196
Patients with any eligible Part III 101: 325


## 8. Calculate complete GDS-15 totals

Reverse-score the five positively worded items and calculate `GDS_TOTAL` only when all 15 responses are present. A partial form is not presented as a complete score.

In [22]:
gds = numeric_fields(
    eligible_sources["gds15"],
    source_specs["gds15"]["fields"],
)
gds_items = source_specs["gds15"]["fields"]
gds_reverse = [
    "GDSSATIS", "GDSGSPIR", "GDSHAPPY", "GDSALIVE", "GDSENRGY",
]

scored_gds = gds[gds_items].copy()
scored_gds[gds_reverse] = 1 - scored_gds[gds_reverse]
gds["GDS_COMPLETE"] = gds[gds_items].notna().all(axis=1).astype(int)
gds["GDS_TOTAL"] = scored_gds.sum(axis=1).where(gds["GDS_COMPLETE"].eq(1))
cleaned["gds15"] = gds

gds_response_counts = gds[gds_items].notna().sum(axis=1)
partial_gds = gds_response_counts.between(1, 14)

print("Partial forms without a total:", int(partial_gds.sum()))
print("Patients with a partial form:", gds.loc[partial_gds, "PATNO"].nunique())

Partial forms without a total: 13
Patients with a partial form: 11


## 9. Clean BMI using eligible history only

Define the approved consensus-height helper. It uses only measurements already shown to be before the patient's landmark and returns missing when the history does not support one clear height.

In [23]:
def height_consensus(group):
    """Return a supported adult height from one patient's eligible history."""
    heights = group["HTCM_RAW"].dropna()
    plausible = heights[heights.between(120, 210)]

    if plausible.empty:
        return np.nan
    if plausible.max() - plausible.min() <= 10:
        return float(plausible.median())

    clusters = (plausible / 5).round() * 5
    counts = clusters.value_counts()
    leaders = counts[counts.eq(counts.max())].index

    if len(leaders) == 1 and counts.iloc[0] >= 2:
        return float(plausible.loc[clusters.eq(leaders[0])].median())

    candidates = {
        float(cluster): float(plausible.loc[clusters.eq(cluster)].median())
        for cluster in clusters.unique()
    }
    weights = group["WGTKG_RAW"].dropna()
    scores = {
        cluster: int((weights / (height / 100) ** 2).between(10, 60).sum())
        for cluster, height in candidates.items()
    }
    best_score = max(scores.values())
    winners = [
        cluster
        for cluster, score in scores.items()
        if score == best_score
    ]
    return candidates[winners[0]] if len(winners) == 1 else np.nan

Convert the eligible vital-sign fields and calculate one consensus height for each patient without using later visits.

In [24]:
vitals = numeric_fields(
    eligible_sources["vital_signs"],
    source_specs["vital_signs"]["fields"],
)
vitals["HTCM_RAW"] = vitals["HTCM"]
vitals["WGTKG_RAW"] = vitals["WGTKG"]

consensus_height = vitals.groupby("PATNO", sort=False).apply(
    height_consensus,
    include_groups=False,
)
vitals["HEIGHT_CONSENSUS_CM"] = vitals["PATNO"].map(consensus_height)

Correct a clearly inconsistent height only when the eligible history supplies a supported replacement. Preserve an explicit flag when a conflict remains unresolved.

In [25]:
plausible_height = vitals["HTCM_RAW"].between(120, 210)
height_spread = (
    vitals.groupby("PATNO")["HTCM_RAW"].transform("max")
    - vitals.groupby("PATNO")["HTCM_RAW"].transform("min")
)
vitals["HEIGHT_CONFLICT_UNRESOLVED"] = (
    height_spread.gt(10)
    & vitals["HEIGHT_CONSENSUS_CM"].isna()
).astype(int)

height_disagrees = (
    vitals["HTCM_RAW"].notna()
    & vitals["HEIGHT_CONSENSUS_CM"].notna()
    & vitals["HTCM_RAW"].sub(vitals["HEIGHT_CONSENSUS_CM"]).abs().gt(10)
)
height_outside = vitals["HTCM_RAW"].notna() & ~plausible_height

vitals["HEIGHT_CORRECTED"] = (
    (height_disagrees | height_outside)
    & vitals["HEIGHT_CONSENSUS_CM"].notna()
).astype(int)
vitals["HTCM_CLEAN"] = vitals["HTCM_RAW"].where(
    ~vitals["HEIGHT_CORRECTED"].eq(1),
    vitals["HEIGHT_CONSENSUS_CM"],
)
vitals.loc[
    height_outside & vitals["HEIGHT_CONSENSUS_CM"].isna(),
    "HTCM_CLEAN",
] = np.nan

Invalidate a weight only when it differs substantially from the patient's eligible history, then calculate BMI and leave unresolved or implausible values missing. No patient is removed and no cohort mean is inserted.

In [26]:
provisional_bmi = vitals["WGTKG_RAW"] / (vitals["HTCM_CLEAN"] / 100) ** 2

typical_weight = vitals["WGTKG_RAW"].where(
    provisional_bmi.between(15, 40)
    & vitals["HEIGHT_CONFLICT_UNRESOLVED"].eq(0)
).groupby(vitals["PATNO"]).transform("median")
typical_weight = typical_weight.fillna(
    vitals.groupby("PATNO")["WGTKG_RAW"].transform("median")
)

vitals["WEIGHT_INVALIDATED"] = (
    vitals["WGTKG_RAW"].notna()
    & typical_weight.notna()
    & vitals["WGTKG_RAW"].sub(typical_weight).abs().div(typical_weight).gt(0.50)
).astype(int)
vitals["WGTKG_CLEAN"] = vitals["WGTKG_RAW"].where(
    vitals["WEIGHT_INVALIDATED"].eq(0)
)

clean_bmi = vitals["WGTKG_CLEAN"] / (vitals["HTCM_CLEAN"] / 100) ** 2
vitals["BMI_OUTSIDE_10_60"] = (
    clean_bmi.notna() & ~clean_bmi.between(10, 60)
).astype(int)
vitals["BMI"] = clean_bmi.where(clean_bmi.between(10, 60))
cleaned["vital_signs"] = vitals

Summarize BMI corrections as aggregate record and patient counts.

In [27]:
bmi_summary = pd.DataFrame([
    {
        "check": "Height records corrected",
        "records": int(vitals["HEIGHT_CORRECTED"].sum()),
        "patients": vitals.loc[vitals["HEIGHT_CORRECTED"].eq(1), "PATNO"].nunique(),
    },
    {
        "check": "Patients with unresolved height conflict",
        "records": pd.NA,
        "patients": vitals.loc[
            vitals["HEIGHT_CONFLICT_UNRESOLVED"].eq(1), "PATNO"
        ].nunique(),
    },
    {
        "check": "Weight records invalidated",
        "records": int(vitals["WEIGHT_INVALIDATED"].sum()),
        "patients": vitals.loc[vitals["WEIGHT_INVALIDATED"].eq(1), "PATNO"].nunique(),
    },
    {
        "check": "Calculated BMI outside 10–60",
        "records": int(vitals["BMI_OUTSIDE_10_60"].sum()),
        "patients": vitals.loc[vitals["BMI_OUTSIDE_10_60"].eq(1), "PATNO"].nunique(),
    },
    {
        "check": "Clean BMI available",
        "records": int(vitals["BMI"].notna().sum()),
        "patients": vitals.loc[vitals["BMI"].notna(), "PATNO"].nunique(),
    },
])

display(bmi_summary)

,check,records,patients
0,Height records corrected,23,20
1,Patients with unresolved height conflict,<NA>,9
2,Weight records invalidated,11,11
3,Calculated BMI outside 10–60,0,0
4,Clean BMI available,4760,1040


## 10. Build the cleaned visit-level outputs

Keep identifiers and timing fields required for reproducible aggregation, cleaned predictor values, and explicit cleaning flags. Outcome fields and follow-up dates are excluded.

In [28]:
metadata_order = [
    "REC_ID", "PATNO", "EVENT_ID", "PAG_NAME", "INFODT", "DATE", "index_date",
]
output_frames = {}

for source_name, frame in cleaned.items():
    spec = source_specs[source_name]
    fields = list(spec["fields"])

    if source_name == "gds15":
        fields += ["GDS_COMPLETE", "GDS_TOTAL"]
    if source_name == "updrs_part_iii":
        fields += part_iii_flags
    if source_name == "vital_signs":
        fields = [
            "HTCM_RAW", "WGTKG_RAW", "HTCM_CLEAN", "WGTKG_CLEAN", "BMI",
            "SYSSUP", "DIASUP", "HRSUP", "SYSSTND", "DIASTND", "HRSTND",
            "HEIGHT_CONSENSUS_CM", "HEIGHT_CORRECTED",
            "HEIGHT_CONFLICT_UNRESOLVED", "WEIGHT_INVALIDATED",
            "BMI_OUTSIDE_10_60",
        ]

    requested = metadata_order + spec.get("extra_metadata", []) + fields
    keep = [column for column in requested if column in frame.columns]
    output_frames[source_name] = frame[keep].copy()

Show the size of every cleaned table. These summaries confirm coverage without revealing patient-level values.

In [29]:
output_summary = pd.DataFrame([
    {
        "table": source_name,
        "records": len(frame),
        "patients": frame["PATNO"].nunique(),
        "columns": frame.shape[1],
    }
    for source_name, frame in output_frames.items()
])

display(output_summary)

,table,records,patients,columns
0,demographics,1043,1043,16
1,family_history,1193,1021,8
2,diagnosis_history,1043,1043,13
3,dopamine_therapy,1245,838,8
4,other_clinical,7174,1040,9
5,moca,4984,1040,8
6,freezing,2944,977,8
7,scopa_aut,5421,1040,9
8,neuroqol_mobility,2944,976,15
9,updrs_part_i_patient,9396,1040,15


## 11. Validate the scientific cleaning invariants

Check the shared invariants for every output: no rows lost during cleaning, all records remain landmark eligible, record identifiers remain unique, and no outcome field enters a predictor table.

In [30]:
validation_rows = []

for source_name, frame in output_frames.items():
    checks = {
        "row count preserved after cleaning": (
            len(frame) == len(eligible_sources[source_name])
        ),
        "all dates observed and landmark eligible": (
            frame["DATE"].notna().all()
            and frame["DATE"].le(frame["index_date"]).all()
        ),
        "record identifiers unique": frame["REC_ID"].is_unique,
        "outcome fields absent": {
            "FLNFR12M", "falls_raw", "falls_class", "outcome_date",
        }.isdisjoint(frame.columns),
    }
    validation_rows.extend(
        {
            "table": source_name,
            "check": check,
            "result": bool(result),
        }
        for check, result in checks.items()
    )

Add instrument-specific checks for unable-to-rate codes, nominal meanings, GDS completeness, and BMI plausibility.

In [31]:
special_checks = {
    "Part I clinical items contain no 101": all(
        not output_frames["updrs_part_i"][field].eq(101).any()
        for field in part_i_items
    ),
    "Part III clinical fields contain no 101": all(
        not output_frames["updrs_part_iii"][field].eq(101).any()
        for field in ["NP3GAIT", "NP3PSTBL", "NHY"]
    ),
    "Part IV clinical items contain no 101": all(
        not output_frames["updrs_part_iv"][field].eq(101).any()
        for field in part_iv_items
    ),
    "Postural hypotension categories are valid": (
        set(other_clinical["FEATPOSHYP"].dropna().unique())
        <= {"No", "Yes", "Uncertain"}
    ),
    "REM-sleep behavior categories are valid": (
        set(other_clinical["FEATSUGRBD"].dropna().unique())
        <= {"No", "Yes", "Uncertain"}
    ),
    "Incomplete GDS forms have no total": gds.loc[
        gds["GDS_COMPLETE"].eq(0), "GDS_TOTAL"
    ].isna().all(),
    "Complete GDS totals are within 0–15": (
        gds["GDS_TOTAL"].dropna().between(0, 15).all()
    ),
    "Clean BMI is within 10–60": vitals["BMI"].dropna().between(10, 60).all(),
}

validation_rows.extend(
    {"table": "all", "check": check, "result": bool(result)}
    for check, result in special_checks.items()
)

Enforce every validation result before saving. A failed check stops the notebook and displays only the failed check names.

In [32]:
validation = pd.DataFrame(validation_rows)
failed_validation = validation.loc[~validation["result"]]

assert failed_validation.empty, failed_validation
print(f"Validation checks passed: {validation['result'].sum()}/{len(validation)}")

Validation checks passed: 68/68


## 12. Summarize deterministic cleaning changes

Create one compact audit table for the approved corrections. These counts describe data cleaning and never select predictors.

In [33]:
part_i_raw_101 = (
    eligible_sources["updrs_part_i"][part_i_items].eq("101").any(axis=1)
)
part_iv_raw_101 = (
    eligible_sources["updrs_part_iv"][part_iv_items].eq("101").any(axis=1)
)

clinical_change_rows = [
    {
        "rule": "Part I 101 converted to missing",
        "affected_records": sum(part_i_101_counts.values()),
        "affected_patients": part_i.loc[part_i_raw_101, "PATNO"].nunique(),
    },
    {
        "rule": "Part III 101 converted to missing with flags",
        "affected_records": sum(part_iii_101_counts.values()),
        "affected_patients": affected_part_iii_patients,
    },
    {
        "rule": "Part IV 101 converted to missing",
        "affected_records": sum(part_iv_101_counts.values()),
        "affected_patients": part_iv.loc[part_iv_raw_101, "PATNO"].nunique(),
    },
    {
        "rule": "Partial GDS-15 left without total",
        "affected_records": int(partial_gds.sum()),
        "affected_patients": gds.loc[partial_gds, "PATNO"].nunique(),
    },
]

Add the preserved categorical states and BMI-history corrections to the same audit table, then display the combined summary.

In [34]:
other_change_rows = [
    {
        "rule": "FEATPOSHYP Uncertain preserved",
        "affected_records": int(other_clinical["FEATPOSHYP"].eq("Uncertain").sum()),
        "affected_patients": other_clinical.loc[
            other_clinical["FEATPOSHYP"].eq("Uncertain"), "PATNO"
        ].nunique(),
    },
    {
        "rule": "FEATSUGRBD Uncertain preserved",
        "affected_records": int(other_clinical["FEATSUGRBD"].eq("Uncertain").sum()),
        "affected_patients": other_clinical.loc[
            other_clinical["FEATSUGRBD"].eq("Uncertain"), "PATNO"
        ].nunique(),
    },
    {
        "rule": "Height corrected from eligible history",
        "affected_records": int(vitals["HEIGHT_CORRECTED"].sum()),
        "affected_patients": vitals.loc[
            vitals["HEIGHT_CORRECTED"].eq(1), "PATNO"
        ].nunique(),
    },
    {
        "rule": "Weight invalidated from eligible history",
        "affected_records": int(vitals["WEIGHT_INVALIDATED"].sum()),
        "affected_patients": vitals.loc[
            vitals["WEIGHT_INVALIDATED"].eq(1), "PATNO"
        ].nunique(),
    },
    {
        "rule": "Records carrying unresolved height flag",
        "affected_records": int((
            vitals["HEIGHT_CONFLICT_UNRESOLVED"].eq(1)
            & (vitals["HTCM_RAW"].notna() | vitals["WGTKG_RAW"].notna())
        ).sum()),
        "affected_patients": vitals.loc[
            vitals["HEIGHT_CONFLICT_UNRESOLVED"].eq(1), "PATNO"
        ].nunique(),
    },
]

change_summary = pd.DataFrame(clinical_change_rows + other_change_rows)
display(change_summary)

,rule,affected_records,affected_patients
0,Part I 101 converted to missing,32,17
1,Part III 101 converted to missing with flags,1196,325
2,Part IV 101 converted to missing,65,33
3,Partial GDS-15 left without total,13,11
4,FEATPOSHYP Uncertain preserved,232,158
5,FEATSUGRBD Uncertain preserved,349,216
6,Height corrected from eligible history,23,20
7,Weight invalidated from eligible history,11,11
8,Records carrying unresolved height flag,26,9


## 13. Save the cleaned tables and audit files

Write the 15 cleaned visit-level tables and their aggregate validation artifacts to the new final-run directory. The original raw CSVs and verified Notebook 22 outputs remain unchanged.

In [35]:
for source_name, frame in output_frames.items():
    frame.to_csv(
        cleaned_directory / f"{source_name}_clean.csv",
        index=False,
    )

temporal_flow.to_csv(output_directory / "cleaning_temporal_flow.csv", index=False)
source_contract.to_csv(output_directory / "cleaning_source_contract.csv", index=False)
change_summary.to_csv(output_directory / "cleaning_change_summary.csv", index=False)
bmi_summary.to_csv(output_directory / "bmi_cleaning_summary.csv", index=False)
validation.to_csv(output_directory / "cleaning_validation.csv", index=False)
output_summary.to_csv(output_directory / "cleaned_table_summary.csv", index=False)

print(f"Saved {len(output_frames)} cleaned tables and six audit files.")

Saved 15 cleaned tables and six audit files.


Compare each new cleaned CSV with the verified Notebook 22 version using SHA-256. Any mismatch blocks aggregation and must be reviewed rather than accepted automatically.

In [36]:
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


comparison_rows = []
for source_name in output_frames:
    new_path = cleaned_directory / f"{source_name}_clean.csv"
    reference_path = reference_directory / f"{source_name}_clean.csv"

    comparison_rows.append({
        "table": source_name,
        "reference_exists": reference_path.exists(),
        "exact_file_match": (
            reference_path.exists()
            and sha256(new_path) == sha256(reference_path)
        ),
        "new_sha256": sha256(new_path),
        "reference_sha256": (
            sha256(reference_path) if reference_path.exists() else pd.NA
        ),
    })

comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(
    output_directory / "verified_output_comparison.csv",
    index=False,
)

assert comparison["reference_exists"].all()
assert comparison["exact_file_match"].all(), comparison.loc[
    ~comparison["exact_file_match"]
]

display(comparison[["table", "exact_file_match"]])
print(f"Exact matches: {comparison['exact_file_match'].sum()}/{len(comparison)} tables")

,table,exact_file_match
0,demographics,True
1,family_history,True
2,diagnosis_history,True
3,dopamine_therapy,True
4,other_clinical,True
5,moca,True
6,freezing,True
7,scopa_aut,True
8,neuroqol_mobility,True
9,updrs_part_i_patient,True


Exact matches: 15/15 tables


## Result required before Notebook 02

All validation checks must pass and all 15 regenerated tables must exactly match the verified cleaning outputs. Notebook 02 may then aggregate these visit-level records into the primary 1,040-patient landmark table and the named sensitivity inputs.

This notebook performs no imputation, statistical feature testing, feature engineering, feature selection, or model fitting.